# ST-GCN Punch Classification Training

Train a Spatial-Temporal Graph Convolutional Network on segmented punch clips to classify boxing punches into 4 classes: jab, cross, hook, uppercut.

**Input:** (T=48, 9 joints, 3 XYZ) 3D skeleton clips
**Output:** Class probability distribution over 4 punch types
**Architecture:** ST-GCN built from scratch in PyTorch
**Validation:** Subject-level split (default: subjects 1-3 train, subject 4 test) with optional 4-fold cross-validation

In [ ]:
import json
import random
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Configuration

Paths and hyperparameters. Adjust these at the top, the rest of the notebook reads from here.

In [ ]:
PROJECT_ROOT = Path("../../")

METADATA_DIR = PROJECT_ROOT / "data" / "metadata" / "no_hardware"
CLIPS_DIR = PROJECT_ROOT / "data" / "clips" / "no_hardware"
ANNOTATIONS_CSV = METADATA_DIR / "annotations.csv"
MODELS_DIR = PROJECT_ROOT / "models" / "stgcn"
LOGS_DIR = PROJECT_ROOT / "logs" / "tensorboard"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

# Data parameters
T = 48
N_JOINTS = 9
N_CHANNELS = 3  # XYZ
N_CLASSES = 5
CLASS_TO_IDX = {"cross": 0, "hook": 1, "jab": 2, "uppercut": 3, "no_punch": 4}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}

# Training hyperparameters
BATCH_SIZE = 32
LEARNING_RATE = 0.001
EPOCHS = 80
WEIGHT_DECAY = 1e-4

# Validation strategy
TEST_SUBJECT = "subject04"  # Used for single-split development mode
RUN_CROSS_VALIDATION = False  # Set True to run full 4-fold CV at the end

# Tensorboard run name
RUN_NAME = f"stgcn_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

print(f"Run name: {RUN_NAME}")
print(f"Logs:     {LOGS_DIR / RUN_NAME}")
print(f"Models:   {MODELS_DIR}")

## Dataset Class

Loads clips from disk, pads to T=48 using last-frame replication, returns (tensor, label).

In [ ]:
class PunchDataset(Dataset):
    """PyTorch Dataset for segmented punch clips.
    
    Loads clips from data/clips/no_hardware/{class}/*.npy and pads to T frames.
    """
    
    def __init__(self, annotations_df: pd.DataFrame, clips_dir: Path, T: int):
        self.df = annotations_df.reset_index(drop=True)
        self.clips_dir = clips_dir
        self.T = T
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        clip_path = self.clips_dir / row["class"] / f"{row['clip_id']}.npy"
        clip = np.load(clip_path).astype(np.float32)  # (frames, 9, 3)
        
        clip = self._pad_or_truncate(clip)  # (T, 9, 3)
        
        # Reshape to ST-GCN expected format: (C, T, V) = (3, T, 9)
        clip = clip.transpose(2, 0, 1)
        
        tensor = torch.from_numpy(clip)
        label = CLASS_TO_IDX[row["class"]]
        
        return tensor, label
    
    def _pad_or_truncate(self, clip: np.ndarray) -> np.ndarray:
        """Pad with last frame to T, or truncate if longer."""
        if clip.shape[0] >= self.T:
            return clip[:self.T]
        
        pad_amount = self.T - clip.shape[0]
        last_frame = clip[-1:].repeat(pad_amount, axis=0)
        return np.concatenate([clip, last_frame], axis=0)


# Quick test
df_all = pd.read_csv(ANNOTATIONS_CSV)
test_ds = PunchDataset(df_all.head(2), CLIPS_DIR, T=T)
sample_tensor, sample_label = test_ds[0]
print(f"Sample tensor shape: {sample_tensor.shape}  (expected: ({N_CHANNELS}, {T}, {N_JOINTS}))")
print(f"Sample label: {sample_label} ({IDX_TO_CLASS[sample_label]})")

## Graph Construction

Define the skeleton adjacency matrix for the 9 joints.

**Joint indices (from `src/constants.py`):**
0: nose, 1: left_shoulder, 2: right_shoulder, 3: left_elbow, 4: right_elbow,
5: left_wrist, 6: right_wrist, 7: left_hip, 8: right_hip

In [ ]:
# Anatomical skeleton edges (joint index pairs)
SKELETON_EDGES = [
    (0, 1), (0, 2),       # nose -> shoulders
    (1, 3), (3, 5),       # left arm: shoulder -> elbow -> wrist
    (2, 4), (4, 6),       # right arm: shoulder -> elbow -> wrist
    (1, 7), (2, 8),       # shoulder -> hip
    (7, 8),               # hip-to-hip
]

def build_adjacency(edges: list, n_joints: int) -> np.ndarray:
    """Build normalized adjacency matrix A_hat = D^-0.5 (A + I) D^-0.5."""
    A = np.eye(n_joints)
    for i, j in edges:
        A[i, j] = 1
        A[j, i] = 1
    
    D = np.sum(A, axis=1)
    D_inv_sqrt = np.diag(1.0 / np.sqrt(D))
    A_hat = D_inv_sqrt @ A @ D_inv_sqrt
    return A_hat

A_hat = build_adjacency(SKELETON_EDGES, N_JOINTS)
print(f"Adjacency matrix shape: {A_hat.shape}")
print(f"Normalized A_hat (rounded):")
print(np.round(A_hat, 2))

# Visualize the skeleton graph
fig, ax = plt.subplots(figsize=(6, 5))
joint_positions = {
    0: (0, 3), 1: (-1, 2), 2: (1, 2),
    3: (-1.5, 1), 4: (1.5, 1),
    5: (-2, 0), 6: (2, 0),
    7: (-0.5, 0.5), 8: (0.5, 0.5),
}
joint_names = ["nose", "L_sh", "R_sh", "L_el", "R_el", "L_wr", "R_wr", "L_hip", "R_hip"]

for i, j in SKELETON_EDGES:
    x = [joint_positions[i][0], joint_positions[j][0]]
    y = [joint_positions[i][1], joint_positions[j][1]]
    ax.plot(x, y, "k-", linewidth=2)

for idx, (x, y) in joint_positions.items():
    ax.scatter(x, y, s=300, c="steelblue", zorder=5)
    ax.annotate(joint_names[idx], (x, y), fontsize=8, ha="center", va="center", color="white", fontweight="bold")

ax.set_title("Skeleton Graph (9 joints)")
ax.set_aspect("equal")
ax.axis("off")
plt.tight_layout()
plt.show()

## ST-GCN Model

Built from scratch following Yan et al. 2018. Each block: graph convolution → temporal convolution → batch norm + ReLU.

In [ ]:
class GraphConv(nn.Module):
    """Spatial graph convolution: X' = A_hat @ X @ W"""
    def __init__(self, in_channels: int, out_channels: int, A_hat: torch.Tensor):
        super().__init__()
        self.register_buffer("A_hat", A_hat)
        self.linear = nn.Linear(in_channels, out_channels)
    
    def forward(self, x):
        # x: (B, C, T, V)
        B, C, T, V = x.shape
        x = x.permute(0, 2, 3, 1)  # (B, T, V, C)
        x = torch.einsum("vu,btuc->btvc", self.A_hat, x)
        x = self.linear(x)
        x = x.permute(0, 3, 1, 2)  # (B, C', T, V)
        return x


class STGCNBlock(nn.Module):
    """Single ST-GCN block: spatial graph conv + temporal conv + residual."""
    def __init__(self, in_channels: int, out_channels: int, A_hat: torch.Tensor,
                 temporal_kernel: int = 9, stride: int = 1):
        super().__init__()
        self.gcn = GraphConv(in_channels, out_channels, A_hat)
        self.tcn = nn.Sequential(
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels,
                      kernel_size=(temporal_kernel, 1),
                      stride=(stride, 1),
                      padding=(temporal_kernel // 2, 0)),
            nn.BatchNorm2d(out_channels),
        )
        # Residual connection
        if in_channels == out_channels and stride == 1:
            self.residual = lambda x: x
        else:
            self.residual = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=(stride, 1)),
                nn.BatchNorm2d(out_channels),
            )
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        res = self.residual(x)
        x = self.gcn(x)
        x = self.tcn(x)
        x = self.relu(x + res)
        return x


class STGCN(nn.Module):
    """Full ST-GCN model for punch classification."""
    def __init__(self, A_hat: torch.Tensor, n_classes: int = 4,
                 in_channels: int = 3, n_joints: int = 9):
        super().__init__()
        self.data_bn = nn.BatchNorm1d(in_channels * n_joints)
        
        self.blocks = nn.ModuleList([
            STGCNBlock(in_channels, 64, A_hat),
            STGCNBlock(64, 64, A_hat),
            STGCNBlock(64, 128, A_hat, stride=2),
            STGCNBlock(128, 128, A_hat),
            STGCNBlock(128, 256, A_hat, stride=2),
            STGCNBlock(256, 256, A_hat),
        ])
        
        self.fc = nn.Linear(256, n_classes)
    
    def forward(self, x):
        # x: (B, C, T, V)
        B, C, T, V = x.shape
        
        # Data BN (normalize per joint)
        x = x.permute(0, 1, 3, 2).contiguous().view(B, C * V, T)
        x = self.data_bn(x)
        x = x.view(B, C, V, T).permute(0, 1, 3, 2)
        
        for block in self.blocks:
            x = block(x)
        
        # Global average pooling over T and V
        x = x.mean(dim=[2, 3])  # (B, C')
        x = self.fc(x)
        return x


# Sanity check
A_hat_tensor = torch.from_numpy(A_hat).float()
model = STGCN(A_hat_tensor, n_classes=N_CLASSES).to(DEVICE)
sample_input = torch.randn(2, N_CHANNELS, T, N_JOINTS).to(DEVICE)
sample_output = model(sample_input)
print(f"Model output shape: {sample_output.shape}  (expected: (2, {N_CLASSES}))")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

## Training Loop

Single-split training. Subjects 1-3 → train, subject 4 → test.
Best model is saved by highest test accuracy.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for inputs, labels in loader:
        inputs = inputs.to(DEVICE)
        labels = labels.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * inputs.size(0)
        _, preds = outputs.max(1)
        correct += (preds == labels).sum().item()
        total += inputs.size(0)
    
    return total_loss / total, correct / total


def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(DEVICE)
            labels = labels.to(DEVICE)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item() * inputs.size(0)
            _, preds = outputs.max(1)
            correct += (preds == labels).sum().item()
            total += inputs.size(0)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    return total_loss / total, correct / total, all_preds, all_labels


def train_model(train_df, test_df, run_name, epochs=EPOCHS):
    """Train one model on the given split. Returns history + best model path."""
    
    train_ds = PunchDataset(train_df, CLIPS_DIR, T=T)
    test_ds = PunchDataset(test_df, CLIPS_DIR, T=T)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    print(f"Train: {len(train_ds)} clips | Test: {len(test_ds)} clips")
    
    A_hat_tensor = torch.from_numpy(A_hat).float()
    model = STGCN(A_hat_tensor, n_classes=N_CLASSES).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()
    
    writer = SummaryWriter(LOGS_DIR / run_name)
    
    best_test_acc = 0
    best_model_path = MODELS_DIR / f"{run_name}_best.pt"
    history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}
    
    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        test_loss, test_acc, _, _ = evaluate(model, test_loader, criterion)
        scheduler.step()
        
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["test_loss"].append(test_loss)
        history["test_acc"].append(test_acc)
        
        writer.add_scalar("Loss/train", train_loss, epoch)
        writer.add_scalar("Loss/test", test_loss, epoch)
        writer.add_scalar("Accuracy/train", train_acc, epoch)
        writer.add_scalar("Accuracy/test", test_acc, epoch)
        writer.add_scalar("LR", scheduler.get_last_lr()[0], epoch)
        
        if test_acc > best_test_acc:
            best_test_acc = test_acc
            torch.save(model.state_dict(), best_model_path)
        
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:3d}/{epochs} | "
                  f"Train loss {train_loss:.4f} acc {train_acc:.3f} | "
                  f"Test loss {test_loss:.4f} acc {test_acc:.3f} | "
                  f"Best {best_test_acc:.3f}")
    
    writer.close()
    print(f"\nBest test accuracy: {best_test_acc:.4f}")
    print(f"Best model saved to: {best_model_path}")
    
    return history, best_model_path, best_test_acc

In [ ]:
df_all = pd.read_csv(ANNOTATIONS_CSV)

train_df = df_all[df_all["subject_id"] != TEST_SUBJECT].reset_index(drop=True)
test_df = df_all[df_all["subject_id"] == TEST_SUBJECT].reset_index(drop=True)

print(f"Train subjects: {sorted(train_df['subject_id'].unique())}")
print(f"Test subject:   {TEST_SUBJECT}")
print()

history, best_model_path, best_acc = train_model(train_df, test_df, RUN_NAME)

## Training Visualizations

Loss and accuracy curves from the single-split run.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

epochs_range = range(1, len(history["train_loss"]) + 1)

axes[0].plot(epochs_range, history["train_loss"], label="Train")
axes[0].plot(epochs_range, history["test_loss"], label="Test")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-entropy loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs_range, history["train_acc"], label="Train")
axes[1].plot(epochs_range, history["test_acc"], label="Test")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Confusion Matrix

Evaluate the best model on the test set and plot the confusion matrix.

In [ ]:
A_hat_tensor = torch.from_numpy(A_hat).float()
model = STGCN(A_hat_tensor, n_classes=N_CLASSES).to(DEVICE)
model.load_state_dict(torch.load(best_model_path))

test_ds = PunchDataset(test_df, CLIPS_DIR, T=T)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)
criterion = nn.CrossEntropyLoss()

_, test_acc, all_preds, all_labels = evaluate(model, test_loader, criterion)

class_names = [IDX_TO_CLASS[i] for i in range(N_CLASSES)]
cm = confusion_matrix(all_labels, all_preds, labels=list(range(N_CLASSES)))

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names, ax=ax, cbar=False)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(f"Confusion Matrix — Test Accuracy: {test_acc:.4f}")
plt.tight_layout()
plt.show()

# Per-class accuracy
print("\nPer-class accuracy:")
for i, name in enumerate(class_names):
    class_total = (np.array(all_labels) == i).sum()
    class_correct = ((np.array(all_labels) == i) & (np.array(all_preds) == i)).sum()
    if class_total > 0:
        print(f"  {name}: {class_correct/class_total:.3f}  ({class_correct}/{class_total})")

## 4-Fold Cross-Validation (Optional)

Set `RUN_CROSS_VALIDATION = True` in the configuration cell to enable.
Each subject takes turns as the held-out test set.
Final accuracy is the mean across folds.

In [ ]:
if RUN_CROSS_VALIDATION:
    subjects = sorted(df_all["subject_id"].unique())
    print(f"Running {len(subjects)}-fold cross-validation across subjects: {subjects}\n")
    
    fold_results = []
    
    for fold_idx, test_subj in enumerate(subjects):
        print("=" * 60)
        print(f"FOLD {fold_idx + 1}/{len(subjects)} — Test subject: {test_subj}")
        print("=" * 60)
        
        train_df_fold = df_all[df_all["subject_id"] != test_subj].reset_index(drop=True)
        test_df_fold = df_all[df_all["subject_id"] == test_subj].reset_index(drop=True)
        
        fold_run_name = f"{RUN_NAME}_fold{fold_idx+1}_{test_subj}"
        history_f, model_path_f, best_acc_f = train_model(
            train_df_fold, test_df_fold, fold_run_name, epochs=EPOCHS
        )
        
        # Confusion matrix per fold
        A_hat_tensor = torch.from_numpy(A_hat).float()
        m = STGCN(A_hat_tensor, n_classes=N_CLASSES).to(DEVICE)
        m.load_state_dict(torch.load(model_path_f))
        test_loader_f = DataLoader(
            PunchDataset(test_df_fold, CLIPS_DIR, T=T),
            batch_size=BATCH_SIZE, shuffle=False
        )
        _, acc_f, preds_f, labels_f = evaluate(m, test_loader_f, nn.CrossEntropyLoss())
        cm_f = confusion_matrix(labels_f, preds_f, labels=list(range(N_CLASSES)))
        
        fold_results.append({
            "fold": fold_idx + 1,
            "test_subject": test_subj,
            "best_acc": best_acc_f,
            "confusion_matrix": cm_f,
        })
    
    # Aggregate results
    accs = [r["best_acc"] for r in fold_results]
    mean_acc = np.mean(accs)
    std_acc = np.std(accs)
    
    print("\n" + "=" * 60)
    print("CROSS-VALIDATION SUMMARY")
    print("=" * 60)
    for r in fold_results:
        print(f"  Fold {r['fold']} ({r['test_subject']}): {r['best_acc']:.4f}")
    print(f"\n  Mean: {mean_acc:.4f} ± {std_acc:.4f}")
    
    # Combined confusion matrix
    cm_combined = np.sum([r["confusion_matrix"] for r in fold_results], axis=0)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm_combined, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, ax=ax, cbar=False)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"Combined Confusion Matrix — Mean Acc: {mean_acc:.4f} ± {std_acc:.4f}")
    plt.tight_layout()
    plt.show()
else:
    print("Cross-validation skipped. Set RUN_CROSS_VALIDATION = True in Cell 4 to enable.")

## Summary

Document the final results for the paper.

In [ ]:
print("=" * 60)
print("TRAINING SUMMARY")
print("=" * 60)
print(f"Run name:           {RUN_NAME}")
print(f"Model architecture: ST-GCN ({sum(p.numel() for p in STGCN(torch.from_numpy(A_hat).float()).parameters()):,} params)")
print(f"Total clips:        {len(df_all)}")
print(f"Classes:            {sorted(df_all['class'].unique())}")
print(f"Subjects:           {sorted(df_all['subject_id'].unique())}")
print(f"T window:           {T}")
print(f"Batch size:         {BATCH_SIZE}")
print(f"Learning rate:      {LEARNING_RATE}")
print(f"Epochs:             {EPOCHS}")
print(f"\nSingle-split (test={TEST_SUBJECT}): {best_acc:.4f}")

if RUN_CROSS_VALIDATION:
    print(f"4-fold CV mean:     {mean_acc:.4f} ± {std_acc:.4f}")
print("=" * 60)